In [ ]:
# ===============================
# Artificial Intelligence & NN
# Task 1 – MNIST Classification
# ===============================

import numpy as np
import matplotlib.pyplot as plt

from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

# -----------------------------
# 1) Load and preprocess MNIST
# -----------------------------

(X_train, y_train), (X_test, y_test) = mnist.load_data()

# Normalize pixel values to [0, 1]
X_train = X_train.astype("float32") / 255.0
X_test = X_test.astype("float32") / 255.0

# Flatten images (28x28 -> 784)
X_train = X_train.reshape(-1, 784)
X_test = X_test.reshape(-1, 784)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

# -----------------------------
# 2) Model builder function
# -----------------------------

def build_model(hidden_activation: str, use_dropout: bool = True):
    model = Sequential()

    # Input + Hidden Layer 1
    model.add(Dense(256, activation=hidden_activation, input_shape=(784,)))
    if use_dropout:
        model.add(Dropout(0.2))

    # Hidden Layer 2
    model.add(Dense(256, activation=hidden_activation))
    if use_dropout:
        model.add(Dropout(0.2))

    # Hidden Layers 3–6
    model.add(Dense(128, activation=hidden_activation))
    model.add(Dense(128, activation=hidden_activation))
    model.add(Dense(64, activation=hidden_activation))
    model.add(Dense(32, activation=hidden_activation))

    # Output Layer
    model.add(Dense(10, activation="softmax"))

    # Compile model
    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

# -----------------------------
# 3) Train and Evaluate
# -----------------------------

def train_and_evaluate(hidden_activation: str, epochs: int = 10, batch_size: int = 128):

    print("\n====================================")
    print(f"Training model with activation: {hidden_activation}")
    print("====================================")

    model = build_model(hidden_activation=hidden_activation, use_dropout=True)

    history = model.fit(
        X_train, y_train,
        validation_split=0.1,
        epochs=epochs,
        batch_size=batch_size,
        verbose=1
    )

    test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)

    print(f"\nTest Loss: {test_loss:.4f}")
    print(f"Test Accuracy: {test_acc:.4f}")

    # Predictions
    y_prob = model.predict(X_test, verbose=0)
    y_pred = np.argmax(y_prob, axis=1)

    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, digits=4))

    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot()
    plt.title(f"Confusion Matrix - {hidden_activation}")
    plt.show()

    # Accuracy curves
    plt.figure()
    plt.plot(history.history["accuracy"], label="train_acc")
    plt.plot(history.history["val_accuracy"], label="val_acc")
    plt.title(f"Accuracy - {hidden_activation}")
    plt.legend()
    plt.show()

    # Loss curves
    plt.figure()
    plt.plot(history.history["loss"], label="train_loss")
    plt.plot(history.history["val_loss"], label="val_loss")
    plt.title(f"Loss - {hidden_activation}")
    plt.legend()
    plt.show()

    return test_acc


# -----------------------------
# 4) Run Experiments
# -----------------------------

acc_sigmoid = train_and_evaluate("sigmoid", epochs=10)
acc_tanh = train_and_evaluate("tanh", epochs=10)

print("\n===== Final Comparison =====")
print("Sigmoid Accuracy:", acc_sigmoid)
print("Tanh Accuracy:", acc_tanh)
